In [ ]:
!pip install edgartools sentence-transformers nltk pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.1 MB/s eta 0:00:00


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from edgar import Company, set_identity

def fetch_filing(ticker: str, year: int, form: str, section_name: str):
    set_identity("analyst@edgarlytics.com")

    company = Company(ticker)
    filings = company.get_filings(form=form)

    for filing in filings:
        try:
            filing_year = filing.filing_date.year

            if filing_year != year:
                continue

            parsed = filing.parse()
            if parsed is None:
                continue

            section = parsed.get_sec_section(section_name)

            if section is None:
                return {
                    "year": year,
                    "ticker": ticker,
                    "status": "missing_section"
                }

            return {
                "year": year,
                "ticker": ticker,
                "status": "success",
                "filing": section
            }

        except Exception as e:
            print(f"Error: {ticker} {year} → {e}")
            continue

    return {
        "year": year,
        "ticker": ticker,
        "status": "filing_not_found"
    }

In [ ]:
def process_filing(args):
    company_name, filing = args

    try:
        parsed = filing.parse()
        if parsed is None:
            return None

        risks_section = parsed.get_sec_section("part_i_item_1a")
        if risks_section is None:
            return None

        year = filing.filing_date.year

        return (company_name, year, risks_section)

    except Exception as e:
        print(f"Error processing {company_name}: {e}")
        return None

In [ ]:
import multiprocessing as mp

def fetch_wrapper(args):
    ticker, year, form, section_name = args
    return fetch_filing(ticker, year, form, section_name)


tickers = ["AAPL"]
years = [2024, 2023]

form = "10-K"
section_name = "part_i_item_1a"

tasks = [
    (ticker, year, form, section_name)
    for ticker in tickers
    for year in years
]

with mp.Pool(min(4, mp.cpu_count())) as pool:
    results = pool.map(fetch_wrapper, tasks)

# Keep only successful results
results = [r for r in results if r["status"] == "success"]

print("Valid filings:", len(results))

Valid filings: 2


In [ ]:
for r in results:
    print("\n---")
    print("Year:", r["year"])
    print("Status:", r["status"])
    print("Text length:", len(r.get("filing", "")))


---
Year: 2024
Status: success
Text length: 68887

---
Year: 2023
Status: success
Text length: 67998


In [ ]:
for r in results:
    print("\n" + "="*80)
    print("Year:", r["year"])
    print("\nSTART OF TEXT:\n")
    print(r["filing"][:1000])


Year: 2024

START OF TEXT:

Item 1A.    Risk Factors

The Company’s business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the Company’s business, reputation, results of operations, financial condition and stock price can be materially and adversely affected.

Because of the following factors, as well as other factors affecting the Company’s results of operations and financial condition, past financial performance should not be considered to be a reliable indicator of future performance, and investors should not use historical trends to anticipate results or trends in future periods. This discussion of risk factors contains forward-looking statements.

This section should be read in conjunction with Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Result

In [ ]:
import re

def preprocess_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
from nltk.tokenize import sent_tokenize

def chunk_text(text, max_sentences=4):
    sentences = sent_tokenize(text)

    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences])

        if len(chunk) > 100 and "risk factors" not in chunk.lower():
            chunks.append(chunk)

    return chunks

In [ ]:
from collections import defaultdict

def prepare_yearwise_chunks(results):
    year_chunks = defaultdict(list)

    for item in results:
        year = item["year"]
        text = preprocess_text(item["filing"])
        chunks = chunk_text(text)

        year_chunks[year].extend(chunks)

    return dict(year_chunks)

year_chunks = prepare_yearwise_chunks(results)
years_sorted = sorted(year_chunks.keys(), reverse=True)

for y in years_sorted:
    print(y, "chunks:", len(year_chunks[y]))

2024 chunks: 77
2023 chunks: 76


In [ ]:
from sentence_transformers import SentenceTransformer

models = {
    "minilm": SentenceTransformer("all-MiniLM-L6-v2"),
    "mpnet": SentenceTransformer("sentence-transformers/all-mpnet-base-v2"),
    "bge": SentenceTransformer("BAAI/bge-base-en")
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def embed_chunks(model, chunks):
    return model.encode(
        chunks,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True
    )

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances

def compute_similarity(emb_n, emb_n1):
    cosine_sim = np.dot(emb_n, emb_n1.T)
    euclidean_dist = euclidean_distances(emb_n, emb_n1)

    return cosine_sim, euclidean_dist

In [ ]:
def normalize_euclidean(dist_matrix):
    min_d = dist_matrix.min()
    max_d = dist_matrix.max()

    norm = (dist_matrix - min_d) / (max_d - min_d + 1e-9)
    similarity = 1 - norm

    return similarity

In [ ]:
def build_match_dataframe(chunks_n, chunks_n1, emb_n, emb_n1, model_name):
    cosine_sim, euclidean_dist = compute_similarity(emb_n, emb_n1)
    euclidean_sim = normalize_euclidean(euclidean_dist)

    rows = []

    for i in range(len(chunks_n)):
        cos_scores = cosine_sim[i]
        euc_scores = euclidean_sim[i]

        best_idx = cos_scores.argmax()

        rows.append({
            "model": model_name,
            "risk_year_n": chunks_n[i],
            "risk_year_n1": chunks_n1[best_idx],
            "cosine_score": float(cos_scores[best_idx]),
            "euclidean_score": float(euc_scores[best_idx])
        })

    return rows

In [ ]:
import pandas as pd

all_rows = []

y1, y2 = years_sorted[0], years_sorted[1]

chunks_n = year_chunks[y1]
chunks_n1 = year_chunks[y2]

for model_name, model in models.items():
    print(f"\nProcessing model: {model_name}")

    emb_n = embed_chunks(model, chunks_n)
    emb_n1 = embed_chunks(model, chunks_n1)

    rows = build_match_dataframe(
        chunks_n, chunks_n1, emb_n, emb_n1, model_name
    )

    all_rows.extend(rows)

df = pd.DataFrame(all_rows)


Processing model: minilm


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Processing model: mpnet


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Processing model: bge


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

all_rows = []

y1, y2 = years_sorted[0], years_sorted[1]

chunks_n = year_chunks[y1]
chunks_n1 = year_chunks[y2]

for model_name, model in models.items():
    print(f"\nProcessing model: {model_name}")

    emb_n = embed_chunks(model, chunks_n)
    emb_n1 = embed_chunks(model, chunks_n1)

    rows = build_match_dataframe(
        chunks_n, chunks_n1, emb_n, emb_n1, model_name
    )

    all_rows.extend(rows)

df = pd.DataFrame(all_rows)


Processing model: minilm


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Processing model: mpnet


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Processing model: bge


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
def sample_by_rank(df, n_each=10):
    df_sorted = df.sort_values("cosine_score", ascending=False).reset_index(drop=True)

    top = df_sorted.head(n_each)
    bottom = df_sorted.tail(n_each)

    mid_start = len(df_sorted)//2 - n_each//2
    middle = df_sorted.iloc[mid_start:mid_start+n_each]

    return pd.concat([top, middle, bottom])

In [ ]:
final_samples = []

for model_name in df["model"].unique():
    subset = df[df["model"] == model_name]
    sampled = sample_by_rank(subset, n_each=10)
    final_samples.append(sampled)

final_df = pd.concat(final_samples).reset_index(drop=True)

In [ ]:
def assign_rank(score):
    if score > 0.8:
        return "High"
    elif score > 0.6:
        return "Moderate"
    else:
        return "Low"

In [ ]:
final_df["cosine_rank"] = final_df["cosine_score"].apply(assign_rank)
final_df["euclidean_rank"] = final_df["euclidean_score"].apply(assign_rank)

final_df["human_label"] = ""

In [ ]:
final_df.to_csv("apple_embedding_risk_similarity_evaluation.csv", index=False)
print("CSV saved!")

CSV saved!


In [ ]:
from google.colab import files
files.download("apple_embedding_risk_similarity_evaluation.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>